# Multi-start gallery — all 19 trained patches side by side

Each seed initializes the patch from a different random point in the optimization landscape and trains for 30 epochs. This notebook shows all the resulting patches and their training curves so the best one can be picked visually + by metrics.

In [ ]:
import json, re, math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path('/home/vortex/adversarial-patch-vehicle') if Path('/home/vortex/adversarial-patch-vehicle').exists() else Path.cwd()
MULTI_ROOT = REPO_ROOT / 'experiments/yolo_attack/multi_20260609_100519'
seed_dirs = sorted([d for d in MULTI_ROOT.iterdir() if d.is_dir() and d.name.startswith('seed')], key=lambda d: int(re.findall(r'seed(\d+)', d.name)[0]))
print(f'Found {len(seed_dirs)} seed dirs')

In [ ]:
# Gallery: one tile per patch, labeled with the final val_veh_score
patches = []
final_metrics = []
for d in seed_dirs:
    img_path = d / 'patch_final.png'
    log_path = d / 'train.log'
    if not img_path.exists() or not log_path.exists():
        continue
    seed = int(re.findall(r'seed(\d+)', d.name)[0])
    text = log_path.read_text()
    last_ep_lines = re.findall(r'\[ep\s*(\d+)/\d+\]\s+train_loss=([\d\.]+)\s+val_loss=([\d\.]+)\s+val_veh_score=([\d\.]+)', text)
    if not last_ep_lines:
        continue
    last = last_ep_lines[-1]
    final_metrics.append({'seed': seed, 'ep': int(last[0]), 'train_loss': float(last[1]),
                          'val_loss': float(last[2]), 'val_veh_score': float(last[3])})
    patches.append((seed, img_path))

metrics = pd.DataFrame(final_metrics).sort_values('val_veh_score').reset_index(drop=True)
metrics

In [ ]:
# 4 cols x ceil(N/4) rows
import matplotlib.image as mpimg
n = len(patches)
cols = 4
rows = math.ceil(n / cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 3.0))
axes = np.array(axes).reshape(-1)
for ax in axes: ax.axis('off')
score_by_seed = dict(zip(metrics['seed'], metrics['val_veh_score']))
for ax, (seed, img_path) in zip(axes, patches):
    img = mpimg.imread(img_path)
    ax.imshow(img)
    s = score_by_seed.get(seed, float('nan'))
    ax.set_title(f'seed{seed}  val_veh_score={s:.4f}', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Overlay training curves (val_loss and val_veh_score across epochs) for all seeds
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for d in seed_dirs:
    log = d / 'train.log'
    if not log.exists(): continue
    seed = int(re.findall(r'seed(\d+)', d.name)[0])
    text = log.read_text()
    rows = re.findall(r'\[ep\s*(\d+)/\d+\]\s+train_loss=([\d\.]+)\s+val_loss=([\d\.]+)\s+val_veh_score=([\d\.]+)', text)
    if not rows: continue
    ep = [int(r[0]) for r in rows]
    val_loss = [float(r[2]) for r in rows]
    val_score = [float(r[3]) for r in rows]
    axes[0].plot(ep, val_loss, alpha=0.5, label=f'seed{seed}')
    axes[1].plot(ep, val_score, alpha=0.5, label=f'seed{seed}')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('val_loss'); axes[0].set_title('val_loss across seeds')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('val_veh_score'); axes[1].set_title('val_veh_score across seeds (lower = stronger attack)')
axes[0].legend(fontsize=7, ncol=2)
axes[1].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

## Pick your winner

Reading order:
1. Look at the **gallery** above. Patches that look like 'something' (faces / patterns / repeating motifs) tend to outperform pure noise.
2. Cross-reference with the **metrics table** (sorted by `val_veh_score` ascending = stronger attack first).
3. Look at the **curves**: a seed that converged steadily to a low value is more reliable than one that just got lucky in the last epoch.

Once you pick a seed N, point `active_patch.TGA` to it:
```bash
python -m src.yolo_chroma_attack.export_tga \
    --patch experiments/yolo_attack/multi_20260609_100519/seedN/patch_final.pt \
    --out assets/chroma_key/adv_patch_seedN.TGA
cd assets/chroma_key && ln -sf adv_patch_seedN.TGA active_patch.TGA
```